In [ ]:
!pip -q install transformers torchaudio scikit-learn pandas tqdm

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import random

# תיקיית מקור
SOURCE_DIR = "/content/drive/MyDrive/balanced_data"

# תיקיית יעד חדשה ונקייה
OUTPUT_DIR = "/content/drive/MyDrive/dataset_split"

split_ratio = 0.8
random.seed(42)

AUDIO_EXTS = (".wav", ".flac", ".mp3", ".m4a", ".ogg")

# אם תיקיית היעד כבר קיימת - מוחקים כדי ליצור מחדש נקי
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

def split_folder(src_folder, train_folder, test_folder):
    os.makedirs(train_folder, exist_ok=True)
    os.makedirs(test_folder, exist_ok=True)

    files = [
        f for f in os.listdir(src_folder)
        if os.path.isfile(os.path.join(src_folder, f)) and f.lower().endswith(AUDIO_EXTS)
    ]

    random.shuffle(files)
    split_index = int(len(files) * split_ratio)

    train_files = files[:split_index]
    test_files = files[split_index:]

    for f in train_files:
        shutil.copy2(
            os.path.join(src_folder, f),
            os.path.join(train_folder, f)
        )

    for f in test_files:
        shutil.copy2(
            os.path.join(src_folder, f),
            os.path.join(test_folder, f)
        )

# -------- real --------
split_folder(
    os.path.join(SOURCE_DIR, "real"),
    os.path.join(OUTPUT_DIR, "train", "real"),
    os.path.join(OUTPUT_DIR, "test", "real")
)

# -------- fake --------
fake_root = os.path.join(SOURCE_DIR, "fake")

attack_types = [
    d for d in os.listdir(fake_root)
    if os.path.isdir(os.path.join(fake_root, d))
]

for attack in attack_types:
    src = os.path.join(fake_root, attack)

    train_dest = os.path.join(OUTPUT_DIR, "train", "fake", attack)
    test_dest  = os.path.join(OUTPUT_DIR, "test", "fake", attack)

    split_folder(src, train_dest, test_dest)

print("Split completed successfully!")
print("New dataset created at:", OUTPUT_DIR)

In [ ]:
print("Detected attack types:", attack_types)